# Phase 3: Stage 2, θ_b (mixed 50/50) and θ_en (pure English)

**Only after Gate 2 is approved.** Upload the latest `colab/code_bundle.tar.gz` to `MyDrive/unlearn/`.
Both runs start from θ_ref = `checkpoints/ref_long_s{SEED}.pt` with a fresh optimizer.
- θ_b: 32 PT + 32 EN sequences per step, 1800 steps (`configs/stage2_mixed.yaml`).
- θ_en: 64 EN sequences per step, 500 steps (`configs/stage2_en.yaml`).

Every cell can be re-run after a disconnect: training resumes from its last checkpoint and finished
steps are skipped.

In [ ]:
# 1. Settings + GPU check
SEED = 0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive. MyDrive/unlearn/{checkpoints,results} is the working copy from Phase 3 on.
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/unlearn'
import os
for d in ('checkpoints', 'results', 'gate_packs'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print(sorted(os.listdir(DRIVE)))

In [ ]:
# 3. Data bundle (unchanged since Phase 0) + latest code bundle on top; check the data SHAs;
#    link checkpoints/ and results/ to Drive so every script writes there directly.
!rm -rf /content/unlearn && mkdir -p /content/unlearn
!tar xzf {DRIVE}/unlearn_bundle.tar.gz -C /content/unlearn
!tar xzf {DRIVE}/code_bundle.tar.gz -C /content/unlearn
%cd /content/unlearn
!cd data/processed && sha256sum -c --quiet MANIFEST.sha256 && echo "data OK"
!rm -rf checkpoints results && ln -s {DRIVE}/checkpoints checkpoints && ln -s {DRIVE}/results results

In [ ]:
# 4. Dependencies + tests. tests/test_data.py is not run here: it needs the *_titles.txt files, which are
#    not in the bundle. It passed on the Mac at Gate 0, and cell 3 already proved the .bin files are
#    byte-identical to the Mac copy (SHA-256). Smoke-data tests are skipped (no smoke data on Colab).
!pip -q install pyyaml
!python -m pytest -q --ignore=tests/test_data.py 2>&1 | tail -2

In [ ]:
# 5. Bring the Mac's results into Drive (never overwrites), then re-hash every protected checkpoint.
#    Drive cannot chmod, so this SHA check is what protects θ_ref. It must print OK, never raise.
!python scripts/colab_sync.py merge --src results_mac --dst results
!python scripts/colab_sync.py verify

In [ ]:
# 6. θ_b: mixed Stage 2
![ -f checkpoints/theta_b_s{SEED}.pt ] && echo "theta_b done" || \
  python scripts/train.py --config configs/stage2_mixed.yaml --seed {SEED} --resume 2>&1 | grep -v Warning

In [ ]:
# 7. θ_en: pure-English diagnostic
![ -f checkpoints/theta_en_s{SEED}.pt ] && echo "theta_en done" || \
  python scripts/train.py --config configs/stage2_en.yaml --seed {SEED} --resume 2>&1 | grep -v Warning

In [ ]:
# 8. Evaluate both on validation + train splits (no test splits before Phase 5), validation prompts
import os
for m in ('theta_b', 'theta_en'):
    if os.path.exists(f'results/samples/{m}_s{SEED}.json'):
        print(m, 'already evaluated'); continue
    !python scripts/eval_checkpoint.py --ckpt checkpoints/{m}_s{SEED}.pt --name {m}_s{SEED} \
        --splits pt_val en_val pt_train en_forget_train --prompts val

In [ ]:
# 9. Curves
!python scripts/plots.py curves --runs stage2_mixed_s{SEED} stage2_en_s{SEED}
from IPython.display import Image; Image(f'results/figures/stage2_mixed_s{SEED}_stage2_en_s{SEED}_curves.png')

In [ ]:
# Gate pack: re-check the hashes, then zip results/ (small files only, no checkpoints).
# Download MyDrive/unlearn/gate_packs/gate3_results.zip and unzip it into the project as colab_results/gate3/
!python scripts/colab_sync.py verify
!cd {DRIVE} && rm -f gate_packs/gate3_results.zip && zip -qr gate_packs/gate3_results.zip results && ls -la gate_packs